In [ ]:
%load_ext autoreload
%autoreload 2
from api_wrappers import *
import zipfile
import os

## Using Hugging Face Spaces API for Protein Design

Hugging Face Spaces provide user-friendly interfaces for running machine learning models. However, one of the UI's limitations is that one must be careful not to refresh the page or get disconnected from their session. It is also harder to pipeline multiple models. Hence, every Space automatically exposes API endpoints that can be used to run the models programmatically through the Gradio client. For example. the [RosettaFold3 Space](https://huggingface.co/spaces/hugging-science/RosettaFold3) has its main logic encapsulated in the `/fold_all_jobs` endpoint. Scroll down to the bottom of the Space and click `use via API` to see the doc.

We provide thin wrappers for the Gradio client in `api_wrappers.py` that are demonstrated here but motivate you to look at these functions to get a better sense of how to submit API requests to Hugging Face Spaces directly. This enables you to access more information about the job status and results.

We here showcase an example pipelining RFdiffusion3 > LigandMPNN > RosettaFold3 for a very easy insulin receptor binder.

In [ ]:
HF_TOKEN= "...hf token here..." #add a token for higher queue priority and extended rate limits on your hackathon account

### RFDiffusion3 for backbone generation

In [ ]:
config_file = "./ppi_inputs/ppi_tutorial.yaml"
target_pdb = "./ppi_inputs/4zxb_cropped.pdb"

rfd3_status, rfd3_output_zip = call_rfdiffusion3_space(
    config_file=config_file,
    scaffold_pdb=target_pdb,
    num_batches=1,
    num_designs_per_batch=5,
    extra_args="",  # e.g., "inference_sampler.step_scale=3 inference_sampler.gamma_0=0.2"
    max_duration=300,
    hf_token=HF_TOKEN,  # Add your HF token if using a private space
    output_dir="./ppi_outputs/rfd3"
)

In [ ]:
# Unzip
extracted_dir = "./ppi_outputs/rfd3/extracted"
os.makedirs(extracted_dir, exist_ok=True)
with zipfile.ZipFile(rfd3_output_zip, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

pdb_files = [os.path.join(extracted_dir, f) for f in os.listdir(extracted_dir) if f.endswith('.pdb')]
print(f"Extracted {len(pdb_files)} PDB files:")
for pdb_file in pdb_files:
    print(pdb_file)

### LigandMPNN for sequence design

In [ ]:
ligand_status, ligand_output_zip = call_ligandmpnn_space(
    pdb_files=pdb_files,
    num_batches=1,
    num_designs_per_batch=5,
    chains_to_design="A", #check on rfd3 output PDB which chain is binder, here A
    hf_token=HF_TOKEN,
    output_dir="./ppi_outputs/ligandmpnn"
)

In [ ]:
extracted_dir = "./ppi_outputs/ligandmpnn/extracted"
os.makedirs(extracted_dir, exist_ok=True)
with zipfile.ZipFile(ligand_output_zip, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

pdb_file_dir = os.path.join(extracted_dir, "backbones")
ligandmpnn_pdb_files = [os.path.join(pdb_file_dir, f) for f in os.listdir(pdb_file_dir) if f.endswith('.pdb')]
print(f"Extracted {len(ligandmpnn_pdb_files)} PDB files from ligandpmm outputs:")
for pdb_file in ligandmpnn_pdb_files:
    print(pdb_file)

## RosettaFold3 for structure prediction

In [ ]:
# take only two for faster processing
rf3_status, rf3_output_zip = call_rosettafold3_space(
    job_files=ligandmpnn_pdb_files[:2],
    num_predictions=1,
    early_stopping=0.5,
    diffusion_steps=50,
    hf_token=HF_TOKEN,
    output_dir="./ppi_outputs/rf3"
)

In [ ]:
extracted_dir = "./ppi_outputs/rf3/extracted"
os.makedirs(extracted_dir, exist_ok=True)
with zipfile.ZipFile(rf3_output_zip, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

You can now continue the pipeline by writing functions to process rosettafold3 outputs, use Pymol to align RFD3 backbone and RF3 predictions, etc. This was a small example of how you could combine tools using Hugging Face Spaces API!